## <font color = "plum">**Vision Atom Finding**<font>

### <font color = "navy">**Libraries and Functions**<font>

In [ ]:
import sys
import importlib.metadata
def test_package(package_name):
    """Test if package exists and returns version or -1"""
    try:
        version = importlib.metadata.version(package_name)
    except importlib.metadata.PackageNotFoundError:
        version = '-1'
    return version

if test_package('pyTEMlib') < '0.2025.2.0':
    print('installing pyTEMlib')
    !{sys.executable} -m pip install  --upgrade pyTEMlib -q

    print('done')

# !pip install pyTEMlib==0.2024.9.0

In [ ]:
%matplotlib widget

import numpy as np
import matplotlib.pylab as plt
import os
from SciFiReaders import EMDReader
import tifffile
from matplotlib import cm

# pyTEMlib
import pyTEMlib
import pyTEMlib.file_tools as ft
import pyTEMlib.image_tools as it
from pyTEMlib.atom_tools import atom_refine
from pyTEMlib import graph_tools as gt
from pyTEMlib import crystal_tools as ct

#atom finding
from skimage.feature import blob_log

#clustering
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA
from sklearn.cluster import DBSCAN

In [ ]:
def make_gauss(size_x, size_y, width=1.0, x0=0.0, y0=0.0, intensity=1.0):
    """Make a Gaussian shaped probe """
    size_x = size_x / 2
    size_y = size_y / 2
    x, y = np.mgrid[-size_x:size_x, -size_y:size_y]
    g = np.exp(-((x - x0) ** 2 + (y - y0) ** 2) / 2.0 / width ** 2)
    probe = g / g.sum() * intensity
    return probe


### <font color = "navy">**Data**<font>

In [ ]:
# dark field images 
# ! gdown --id --folder --remaining-ok 1-vplMbLhPO74uyNI5Tc1bGHv-9IH2foH

#oxygen vacancies bright field images 
# ! gdown --id --folder --remaining-ok 1wMQvqWbu6f6yOtm7rH_uOiP_BrUjux1C
# ! gdown --id --folder --remaining-ok 14g911D2X_vrapnqKL-zSBUOvHnvEkzB4

Select a folder

Each folder contain images with different imaging condistions

In [ ]:
path = "/Users/kbarakat/Library/CloudStorage/OneDrive-UniversityofTennessee/vs_code/projects/ybco/vision_/A1-D may/"
folder = path
files = os.listdir(folder)

files = [f for f in files if '.emd' in f]
files = np.sort(files)
names = [f.split(' ')[0] for f in files]

print(len(files))

In [ ]:
for i in range(len(files)):
    dset = ft.open_file(folder + files[i])
    image = dset['Channel_000']
    print(image.shape)

In [ ]:
dset = ft.open_file(folder + files[0])
image1 = dset['Channel_000'][0:512, 0:512]

view = image1.T.plot(cmap = "gray")

In [ ]:
image1.metadata

Normalization

In [ ]:
# avg_im = image.sum(axis=0)
image1.data_type = 'IMAGE'
avg_im = image1
avg_im -= avg_im.min()
avg_im /= avg_im.max()

In [ ]:
print(avg_im.shape)

print(avg_im.min())

In [ ]:
# ------- Input ------
atoms_size = 3 #for stack
# atoms_size = 0.09
# --------------------
# scale = avg_im.x.values[1]

avg_im.metadata['experiment']= {'convergence_angle': 0.028000000000000001, 'acceleration_voltage': 200000.}
# gauss_diameter = atoms_size/scale
gauss_diameter = atoms_size
gauss_probe = make_gauss(avg_im.shape[0], avg_im.shape[1], gauss_diameter)
lr_dataset = it.decon_lr(avg_im, gauss_probe, verbose=False)
inverted = np.log1p(lr_dataset.max() - lr_dataset)

fig, ax = plt.subplots(1,3, figsize=(9, 3), sharex=True, sharey=True)
ax[0].imshow(avg_im.T)
ax[0].set_title('original')
ax[1].imshow(lr_dataset.T)
ax[1].set_title('LR_decon')
ax[2].imshow(inverted.T)
ax[2].set_title('inverted')


Here is the bottleneck
If using bloblog to find atoms we need to fine tune the hyperparameter
* sigma
* threshold

In [ ]:
# ------- Input ------
threshold = .04 #usally between 0.01 and 0.9  the smaller the more atoms
# ----------------------

blobs = blob_log(lr_dataset, max_sigma=5, threshold=threshold)

blobs = pyTEMlib.graph_tools.delete_rim_atoms(blobs, lr_dataset.shape, 10)

fig, ax = plt.subplots(1, 1,figsize=(8,7), sharex=True, sharey=True)
ax.imshow(avg_im, cmap='gray')
ax.scatter(blobs[:, 1], blobs[:, 0], c='r', s=20, alpha = .5)

output_folder = "output"
os.makedirs(output_folder, exist_ok=True)

# save figure
save_path = os.path.join(output_folder, "blob_detection_2.png")
fig.savefig(save_path, dpi=300, bbox_inches="tight")

refine positions

In [ ]:
atom_radius = 2
MaxInt = 0
MinInt = 0
maxDist = 5
sym = atom_refine(np.array(avg_im), blobs, atom_radius, max_int = 0, min_int = 0, max_dist = maxDist)
refined_atoms = np.array(sym['atoms'])


fig, ax = plt.subplots(1, 2, figsize=(8, 4), sharex=True, sharey=True)
ax[0].imshow(avg_im, cmap = 'gray')
ax[0].scatter(refined_atoms[:,1],refined_atoms[:,0],  s=10, alpha = 0.3, color = 'red')
ax[0].set_title('refined atom postion')
ax[1].imshow(avg_im, cmap = 'gray')
ax[1].scatter(blobs[:, 1], blobs[:, 0], c='r', s=10, alpha = .3);

ax[1].set_title('blobs on image');